# Business Entity Resolution: EDA

Thin driver over `src/eval/eda.py` (all logic lives there). Running `run_eda` also rewrites `reports/eda_summary.md` (hand-curated Decisions block preserved) and `reports/eda_examples.md`.

Sampled parts use `sample.frac` / `seed` from `configs/default.yaml`. Runtime about 2 min, peak RAM about 3.4 GB.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import polars as pl
from src.config import load_config, resolve_path
from src.eval import eda

pl.Config.set_tbl_rows(50)
pl.Config.set_tbl_cols(20)
pl.Config.set_fmt_str_lengths(80)
cfg = load_config()

In [2]:
r = eda.run_eda(
    frac=cfg['sample']['frac'],
    seed=cfg['seed'],
    dataset_dir=resolve_path(cfg['paths']['dataset_dir']),
    reports_dir=resolve_path(cfg['paths']['reports_dir']),
)

[   0.0s] row counts


[  22.1s] ground truth


[  29.3s] one-owner / unmatched


[  35.0s] pair text stats


[  45.2s] lengths


[  61.3s] noise patterns


[  79.8s] top tokens


[  90.1s] examples


[ 109.1s] wrote /mnt/D/Business_Entity_Resolution/reports/eda_summary.md and /mnt/D/Business_Entity_Resolution/reports/eda_examples.md


## 1. Row counts, ID checks, test-only countries

In [3]:
print('test-only countries:', r['new_countries'])
r['counts'].pivot(on='source', index=['split', 'country'], values='n').sort('split', 'country')

test-only countries: ['France']


split,country,S1,S2,S3
str,str,u32,u32,u32
"""test""","""France""",259452,703378,731615
"""test""","""India""",809986,2312565,2405000
"""test""","""US""",663106,1871330,1945701
"""train""","""India""",883188,2017799,2115547
"""train""","""US""",1323633,3016817,3170056


In [4]:
r['id_checks']

split,source,n,dup_ids,bad_prefix
str,str,i64,i64,i64
"""train""","""S1""",2206821,0,0
"""train""","""S2""",5034616,0,0
"""train""","""S3""",5285603,0,0
"""test""","""S1""",1732544,0,0
"""test""","""S2""",4887273,0,0
"""test""","""S3""",5082316,0,0


## 2-3. Singleton rate, matches per S1, S2 vs S3

In [5]:
ms = r['match_stats']
print(f"SINGLETON RATE: {ms['singleton_rate']:.4%}  ({ms['n_singleton']:,}/{ms['n_s1']:,})")
print(f"S2 share {ms['share_s2']:.1%}  S3 share {ms['share_s3']:.1%}")
print('coverage:', r['coverage'])
ms['dist']

SINGLETON RATE: 5.5848%  (123,247/2,206,821)
S2 share 48.4%  S3 share 51.6%
coverage: {'s1_missing_from_gt': 0, 'gt_s1_not_in_s1': 0, 'gt_cands_not_in_s23': 0}


matches,n,pct
str,u32,f64
"""0""",123247,5.584821
"""1""",119157,5.399486
"""2""",375212,17.002376
"""3""",530841,24.054556
"""4+""",1058364,47.958761


In [6]:
r['match_by_country']

country,n_s1,pct_singleton,mean_matches,pct_4plus
str,u32,f64,f64,f64
"""India""",883188,5.587825,3.464543,48.061908
"""US""",1323633,5.582816,3.459057,47.889936


## 4. One-owner check

In [7]:
print('S2/S3 ids under 2+ S1s:', r['n_multi_owner'], 'of', r['n_distinct_cands'])
r['multi_owner_examples']

S2/S3 ids under 2+ S1s: 0 of 7638365


cand_id,s1_ids,n_owners
str,list[str],u32


## 5. Unmatched train S2/S3

In [8]:
r['unmatched']

source,country,n,unmatched,pct_unmatched
str,str,u32,u32,f64
"""S2""","""India""",2017799,537254,26.625744
"""S2""","""US""",3016817,803743,26.642087
"""S3""","""India""",2115547,536249,25.348007
"""S3""","""US""",3170056,804608,25.381507


## 6. Matched-pair text stats (sampled S1s)

In [9]:
r['pair_overall']

pairs,%name_exact_lc,%name_tsr<50,tsr_median,%postal_shared,%postal_shared|both_have,%addr_empty_either,%addr_empty_s1,%addr_empty_cand,%cross_country
u32,f64,f64,f32,f64,f64,f64,f64,f64,f64
1529576,10.72088,10.243362,100.0,4.813425,93.222163,4.412988,0.0,4.412988,0.0


In [10]:
r['pair_by']

country,cand_source,pairs,%name_exact_lc,%name_tsr<50,tsr_median,%postal_shared,%postal_shared|both_have,%addr_empty_either,%addr_empty_s1,%addr_empty_cand,%cross_country
str,str,u32,f64,f64,f32,f64,f64,f64,f64,f64,f64
"""India""","""S2""",296658,6.33187,25.589736,93.103447,0.211017,98.119122,3.799662,0.0,3.799662,0.0
"""India""","""S3""",316246,6.740322,16.808434,95.652176,0.214074,98.258345,4.026612,0.0,4.026612,0.0
"""US""","""S2""",442689,14.115101,2.957381,100.0,7.773403,92.85483,4.875884,0.0,4.875884,0.0
"""US""","""S3""",473983,12.953629,3.062979,100.0,7.998177,93.395088,4.622318,0.0,4.622318,0.0


## 7. Length distributions and noise-pattern rates

In [11]:
r['lengths']

split,source,country,n,name_p5,name_p50,name_p95,name_max,addr_p5,addr_p50,addr_p95,addr_max,%name_empty,%addr_empty,%name_nonascii
str,str,str,u32,f64,f64,f64,u32,f64,f64,f64,u32,f64,f64,f64
"""test""","""S1""","""France""",259452,12.0,19.0,29.0,57,40.0,48.0,65.0,147,0.0,0.0,15.721212
"""test""","""S2""","""France""",703378,11.0,20.0,35.0,61,22.0,40.0,59.0,136,0.0,3.061938,24.541143
"""test""","""S3""","""France""",731615,10.0,20.0,36.0,75,22.0,41.0,60.0,139,0.0,2.944308,23.944971
"""test""","""S1""","""India""",809986,13.0,27.0,38.0,92,47.0,76.0,116.0,268,0.0,0.0,0.0
"""test""","""S2""","""India""",2312565,14.0,28.0,44.0,102,31.0,68.0,110.0,269,0.0,2.281622,27.614964
"""test""","""S3""","""India""",2405000,12.0,28.0,44.0,103,20.0,59.0,106.0,267,0.0,2.463202,18.208025
"""test""","""S1""","""US""",663106,11.0,22.0,35.0,66,26.0,34.0,48.0,92,0.0,0.0,0.0
"""test""","""S2""","""US""",1871330,11.0,24.0,39.0,85,22.0,32.0,43.0,75,0.0,2.944804,6.248337
"""test""","""S3""","""US""",1945701,11.0,24.0,41.0,85,26.0,39.0,53.0,107,0.0,2.843037,6.39497


In [12]:
r['noise']

split,source,country,n,%addr_null_literal,%name_domain_like,%name_leading_symbol,%name_non_latin,%name_all_upper,%addr_all_upper,%name_has_brackets,%name_dotted_abbrev
str,str,str,u32,f64,f64,f64,f64,f64,f64,f64,f64
"""test""","""S1""","""France""",259452,0.0,0.0,0.059741,0.0,0.001156,0.0,8.014199,0.003469
"""test""","""S2""","""France""",703378,0.0,3.492432,0.904208,0.0,20.804887,28.976738,10.237738,5.450412
"""test""","""S3""","""France""",731615,0.0,3.43104,0.851677,0.0,5.608004,0.0,10.000615,5.370448
"""test""","""S1""","""India""",809986,0.005062,0.0,0.14778,0.0,0.0,0.0,5.144286,0.036544
"""test""","""S2""","""India""",2312565,2.196652,2.169409,1.514768,23.636352,14.121506,24.000969,9.444837,0.725299
"""test""","""S3""","""India""",2405000,2.130811,2.344574,1.619376,13.332183,2.470478,0.038004,10.061455,0.80973
"""test""","""S1""","""US""",663106,0.0,0.0,0.136931,0.0,0.0,0.000302,0.002262,4.631989
"""test""","""S2""","""US""",1871330,2.914879,3.216643,2.300236,0.0,20.418152,90.479819,6.248764,4.186808
"""test""","""S3""","""US""",1945701,2.803977,3.092356,2.259186,0.0,3.195404,0.0,6.46898,4.172172


## 8. Top-40 last name tokens / address tokens per country

In [13]:
eda._pivot_tokens(r['top_name'], r['k'])

rank,France,India,US
i64,str,str,str
1,"""sarl (17.8%)""","""limited (29.7%)""","""llc (16.0%)"""
2,"""sas (12.7%)""","""ltd (14.2%)""","""inc (11.9%)"""
3,"""eurl (5.1%)""","""लिमिटेड (6.0%)""","""com (3.3%)"""
4,"""sa (4.2%)""","""private (4.8%)""","""corp (3.2%)"""
5,"""sasu (3.7%)""","""llp (3.1%)""","""center (2.9%)"""
6,"""sci (3.2%)""","""com (2.6%)""","""c (2.6%)"""
7,"""com (3.0%)""","""center (2.3%)""","""co (2.6%)"""
8,"""france (2.3%)""","""services (1.8%)""","""group (2.5%)"""
9,"""groupe (1.6%)""","""pvt (1.5%)""","""partners (2.5%)"""


In [14]:
eda._pivot_tokens(r['top_addr'], r['k'])

rank,France,India,US
i64,str,str,str
1,"""de (51.3%)""","""no (44.4%)""","""street (11.0%)"""
2,"""rue (42.9%)""","""road (19.5%)""","""road (10.1%)"""
3,"""la (24.6%)""","""floor (14.8%)""","""drive (8.9%)"""
4,"""r (21.1%)""","""nagar (14.1%)""","""st (8.8%)"""
5,"""loire (19.7%)""","""delhi (13.1%)""","""rd (8.2%)"""
6,"""france (17.3%)""","""c (11.4%)""","""dr (7.4%)"""
7,"""hauts (17.2%)""","""maharashtra (10.4%)""","""avenue (7.3%)"""
8,"""bordeaux (16.4%)""","""a (10.2%)""","""city (6.6%)"""
9,"""nouvelle (14.3%)""","""new (9.3%)""","""ave (6.1%)"""


## 9. Random matched pairs, singleton near-misses

In [15]:
r['ex_pairs']

s1_id,s1_business_name,s1_business_address,cand_id,c_business_name,c_business_address,s1_country
str,str,str,str,str,str,str
"""S1-536747137""","""Saint Cathedral""","""OH, 544 Williams Road, Columbus""","""S2-408369127""","""Saint Cathedral""","""544 WILLIAMS ROAD, COLUMBUS, OH""","""US"""
"""S1-128823504""","""Torres Enterprises LLC""","""3778 Pioneer Road, Marriott-slaterville City, UT""","""S3-654206693""","""Torres LLC Partners""","""Pioneer Road, Ogden Township, Utah""","""US"""
"""S1-842153941""","""Shakti Foundation Private Limited""","""Door No.17/274, 1St Floor, Central Building, Panayi, Ernad, Malappuram, Malapura…","""S2-665280205""","""ശക്തി ഫൗണ്ടേഷൻ പ്രൈവറ്റ് ലിമിറ്റഡ്""","""#17/274, 1ST FLOOR, CENTRAL BUILDING, PANAYI, ERNAD, MALAPURAM, MALAPPURAM, കേരള…","""India"""
"""S1-882818254""","""Page Electronics Corp""","""1110 Jay Street, Griffith, IN""","""S2-48370015""","""... Page C0rp Center""","""#1110 JAY ST, GRIFFITH, IN""","""US"""
"""S1-415075088""","""Infra Vg It Private Limited""","""Plot No.5 Kh No.30/14/1, Matiala Extn., New Delhi, West Delhi, Delhi""","""S2-448462205""","""INFRA VG IT PRIVEAET LIMITED""","""PLOT NO.5 KH NO.30/14/1, MATIALA EXTN., NEW DELHI, दिल्ली""","""India"""
"""S1-402399203""","""DF Immunopharma Inc""","""Unit UP, OH, Fairport Harbor, 401 Fifth Street""","""S2-96967267""","""PARTNERS DF INC""","""01 5TH ST, PAINESVILLE, OH""","""US"""
"""S1-369131297""","""The Dent Tavern""","""7450 32nd Street, Unit APT 1405, Wichita, KS""","""S3-267312705""","""The Dent Tavern Inc""","""32nd St, # APT 1405, Wichita, Kansas""","""US"""
"""S1-270778210""","""Orthopedic Group""","""478 77th Avenue E, Tulsa, OK""","""S2-383261926""","""ORTHOPEDIC""","""TULSA, OK, 77RD AVENUE EAST""","""US"""
"""S1-262978889""","""Titan Red LLC""","""120-08 192 Street, Saint Albans, NY""","""S3-937685820""","""Tan Tan Red LLC""","""New York, ST Lbans, 120-08 192 St""","""US"""


In [16]:
r['ex_singletons']

s1_id,s1_name,s1_addr,country,best_id,best_name,best_addr,score
str,str,str,str,str,str,str,f64
"""S1-536092572""","""Romero, Genevieve, P.A., DDS PC""","""8511 Ridge Run Road, Greenfield Township, PA""","""US""","""S3-363919560""","""Romero, Genevieve, PA, DDS PC Central""","""8520 Ridge Run Rd, <NULL>, Claysbrug, Pennsylvania""",88.5
"""S1-951089518""","""All Engineering Worldwide Inc.""","""513 Frisco Ridge Road, Yukon, OK""","""US""","""S2-156802490""","""All Engineering-Worldwide Inc. Metro""","""522B FRISCO RIDGE RD, YUKON, OK""",90.6
"""S1-11302209""","""Vigilance & Sons Private Limited""","""F-02, 2Nd Floor, Solus, No.2, 1St Cross J C Road, Bengaluru, Bangalore, Karnatak…","""India""","""S2-480139952""","""Vaigai & Sons Private (Limited)""","""Karnataka, NO. 95, 3RD MAIN VINAYAKA LAYOUT NAGARBHAVI 9TH BLOCK, BANGALORE""",87.7
"""S1-201493431""","""Star Tri-State Dte""","""249 Carr Drive, Liberty Hill, TX""","""US""","""S3-806127457""","""Tri-State Star""","""8224 Doss Rd, Chesterfield County, Virginia""",87.5
"""S1-219599078""","""Yoder Management Inc""","""826 100, Unit 6, Spanish Fork, UT""","""US""","""S2-709108597""","""Optdyne Inc Management""","""VINCENNES, NULL, 6991 DEKER ROAD, IN""",85.7
"""S1-72836005""","""Marguerite's Vision Center Corp""","""3212 4800, Corinne, UT""","""US""","""S3-663422397""","""Corp Pardue's Vision Center""","""##1746 Jacinto Cir, MEA, Arizona""",86.2
"""S1-344404581""","""Continental Center LLC""","""6226 Graceland Avenue, Cincinnati, OH""","""US""","""S2-386566089""","""CONTINENTAL LLC CENTER""","""1005 IRISH MOSS LN, MATTHEWS, NC""",100.0
"""S1-556986726""","""Liberty Optimal Technology LLC""","""908 Rosedale Avenue, Rosedale, MD""","""US""","""S2-261047508""","""National Liberty Technologies, Ltd""","""809 CORDAY DR, NULL, NAPERVILLE, IL""",76.2
"""S1-291864389""","""Rector, Elka J., MD Group""","""3435 Cleveland Hill Road, Roseburg, OR""","""US""","""S2-928408416""","""J/E RETURN GROUP""","""55 HILLSMAN LANE, VA, CAMPBELL COUNTY""",73.7


## 10. Test-only country records (20 per source)

In [17]:
for (country, src), g in r['ex_new'].group_by('country', 'source', maintain_order=True):
    print(f'--- {country} {src}')
    print(g.select('entity_id', 'business_name', 'business_address'))

--- France S1
shape: (20, 3)
┌──────────────┬───────────────────────────────┬───────────────────────────────────────────────────┐
│ entity_id    ┆ business_name                 ┆ business_address                                  │
│ ---          ┆ ---                           ┆ ---                                               │
│ str          ┆ str                           ┆ str                                               │
╞══════════════╪═══════════════════════════════╪═══════════════════════════════════════════════════╡
│ S1-9393637   ┆ Centre Médical de Dame        ┆ 24 Rue du Chêne Houpline, Tourcoing,              │
│              ┆                               ┆ Hauts-de-France                                   │
│ S1-756975242 ┆ EEG Sport                     ┆ 90 i Rue de Constantine, Roubaix, Hauts-de-France │
│ S1-295223753 ┆ Amicale du Sports             ┆ 202 La Severie, Pornic, Pays de la Loire          │
│ S1-82897225  ┆ GZ Union SARL                 ┆ 10 Avenue Lou